# FZ Getting Started: Perfect Gas Parametric Study

This notebook walks through **every core fz function** using a classic
thermodynamics example — the ideal gas law:

> **PV = nRT**  →  T = PV / (nR)

We will:
1. `fzl` — inspect installed models/calculators
2. `fzi` — detect variables in an input template
3. `fzc` — compile templates for specific parameter values
4. `fzo` — parse output files
5. `fzr` — run full parametric calculations in parallel

No external simulator needed — a tiny Python script acts as our "solver".


In [1]:
import sys, json, shutil
from pathlib import Path
import fz
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"fz  {fz.__version__}")
fz.set_log_level("WARNING")   # suppress verbose INFO lines in the notebook
fz.print_config()


fz  1.0
FZ PACKAGE CONFIGURATION
Version: 1.0
Commit:  (2026-04-27)

🔧 LOGGING:
  FZ_LOG_LEVEL = WARNING

🔄 CALCULATION:
  FZ_MAX_RETRIES = 5
  FZ_INTERPRETER = python
  Current interpreter = python

⚡ PERFORMANCE:
  FZ_MAX_WORKERS = auto

🌐 SSH:
  FZ_SSH_AUTO_ACCEPT_HOSTKEYS = False
  FZ_SSH_KEEPALIVE = 300s

⏱️  RUN TIMEOUT:
  FZ_RUN_TIMEOUT = 600s

🔍 SHELL PATH:
  FZ_SHELL_PATH = (not set, use system PATH)

Set environment variables to customize these defaults
Example: export FZ_LOG_LEVEL=INFO FZ_MAX_RETRIES=3


## 1 · Workspace setup

In [2]:
WORK = Path("work_01_getting_started")
WORK.mkdir(exist_ok=True)
(WORK / "models").mkdir(exist_ok=True)
print("Workspace:", WORK.resolve())


Workspace: /home/richet/Sync/Open/Funz/github/fz/examples/work_01_getting_started


## 2 · The simulator script

We write a tiny Python script that reads a compiled `params.in` file, applies the ideal gas law, and writes `output.txt`.

In [3]:
SIM = WORK / "models" / "perfectgas.py"
SIM.write_text('''#!/usr/bin/env python3
"""Perfect Gas solver – reads params.in, writes output.txt"""
params = {}
with open("params.in") as fh:
    for line in fh:
        line = line.split("#")[0].strip()
        if "=" in line:
            k, v = line.split("=", 1)
            params[k.strip()] = float(v.strip())

P, V, n = params["P"], params["V"], params["n"]
R = 0.08206          # L·atm / (mol·K)
T = P * V / (n * R)

with open("output.txt", "w") as fh:
    fh.write(f"P = {P}\\n")
    fh.write(f"V = {V}\\n")
    fh.write(f"n = {n}\\n")
    fh.write(f"T = {T:.6f}\\n")
    fh.write("status = OK\\n")
''')
print("Created:", SIM)


Created: work_01_getting_started/models/perfectgas.py


## 3 · Input template

Variables are written as `${name~default}`.  The `~default` part provides a fall-back when no value is passed to `fzc`.

In [4]:
TMPL = WORK / "params.in"
TMPL.write_text(
    "# Perfect Gas input parameters\n"
    "P = ${P~1.013}   # pressure (atm)\n"
    "V = ${V~22.4}    # volume (L)\n"
    "n = ${n~1.0}     # moles of gas\n"
)
print(TMPL.read_text())


# Perfect Gas input parameters
P = ${P~1.013}   # pressure (atm)
V = ${V~22.4}    # volume (L)
n = ${n~1.0}     # moles of gas



## 4 · Model definition

The model tells fz:
- which characters mark variables/formulas
- how to extract output values from result files

In [5]:
MODEL = {
    "varprefix":     "$",
    "formulaprefix": "@",
    "delim":         "{}",
    "commentline":   "#",
    "output": {
        "temperature": "grep '^T = ' output.txt | awk '{print $3}'",
        "pressure":    "grep '^P = ' output.txt | awk '{print $3}'",
    },
}
print(json.dumps(MODEL, indent=2))


{
  "varprefix": "$",
  "formulaprefix": "@",
  "delim": "{}",
  "commentline": "#",
  "output": {
    "temperature": "grep '^T = ' output.txt | awk '{print $3}'",
    "pressure": "grep '^P = ' output.txt | awk '{print $3}'"
  }
}


## 5 · `fzl` — list installed models & calculators

In [6]:
info = fz.fzl()
print("Models:")
for name, props in info["models"].items():
    print(f"  {name}: {props['path']}")
print("\nCalculators:")
for name in info["calculators"]:
    print(f"  {name}")


Models:
  Moret: /home/richet/.fz/models/Moret.json

Calculators:
  sh://


## 6 · `fzi` — detect variables in the template

In [7]:
variables = fz.fzi(str(TMPL), MODEL)
print("Detected variables (name → default):")
print(json.dumps(variables, indent=2))


Detected variables (name → default):
{
  "P": 1.013,
  "V": 22.4,
  "n": 1.0
}


## 7 · `fzc` — compile for a single parameter set

fzc **always** creates a subdirectory named `var1=val1,var2=val2,...` inside the output directory.

In [8]:
def read_compiled(out_dir, filename):
    """Read compiled file – handles both direct and subdirectory layouts."""
    direct = Path(out_dir) / filename
    if direct.exists():
        return direct.read_text()
    subdirs = sorted(d for d in Path(out_dir).iterdir()
                     if d.is_dir() and not d.name.startswith("."))
    if subdirs:
        return (subdirs[0] / filename).read_text()
    raise FileNotFoundError(f"{filename} not found in {out_dir}")

def case_dir(out_dir):
    """Return the first case subdirectory (or out_dir itself for defaults)."""
    p = Path(out_dir)
    subdirs = sorted(d for d in p.iterdir()
                     if d.is_dir() and not d.name.startswith("."))
    return subdirs[0] if subdirs else p

out_single = WORK / "compiled_single"
fz.fzc(str(TMPL), {"P": 2.5, "V": 15.0, "n": 0.75}, MODEL,
       output_dir=str(out_single))
cd = case_dir(out_single)
print(f"Case directory: {cd.name}")
print("Compiled params.in:")
print(cd.read_text() if cd == out_single else (cd / "params.in").read_text())


Case directory: P=2.5,V=15.0,n=0.75
Compiled params.in:
# Perfect Gas input parameters
P = 2.5   # pressure (atm)
V = 15.0    # volume (L)
n = 0.75     # moles of gas



## 8 · `fzc` — compile a Cartesian grid

When `input_variables` contains lists, fzc generates one subdirectory per combination.

In [9]:
out_multi = WORK / "compiled_multi"
fz.fzc(str(TMPL),
       {"P": [1.0, 2.0, 3.0], "V": [10.0, 20.0], "n": [1.0]},
       MODEL,
       output_dir=str(out_multi))

dirs = sorted(d for d in out_multi.iterdir() if d.is_dir() and not d.name.startswith("."))
print(f"Created {len(dirs)} case directories:")
for d in dirs:
    print(f"  {d.name}")


Created 6 case directories:
  P=1.0,V=10.0,n=1.0
  P=1.0,V=20.0,n=1.0
  P=2.0,V=10.0,n=1.0
  P=2.0,V=20.0,n=1.0
  P=3.0,V=10.0,n=1.0
  P=3.0,V=20.0,n=1.0


## 9 · `fzo` — parse outputs (offline)

After the simulator has run, `fzo` harvests output values from directories matching a glob pattern.

In [10]:
import subprocess
# Manually run one case so we have output.txt to parse
case_path = dirs[0]
subprocess.run(["python3", str(SIM.resolve())], cwd=str(case_path), check=True)
print("output.txt preview:")
print((case_path / "output.txt").read_text())

# Parse with fzo
result_fzo = fz.fzo(str(case_path), MODEL)
print("\nfzo result:")
print(result_fzo)


output.txt preview:
P = 1.0
V = 10.0
n = 1.0
T = 121.862052
status = OK


fzo result:
                                                path  temperature  pressure  \
0  work_01_getting_started/compiled_multi/P=1.0,V...   121.862052       1.0   

     P     V    n  
0  1.0  10.0  1.0  


## 10 · `fzr` — full parametric run

In [11]:
CALC = f"sh://python3 {SIM.resolve()}"

df = fz.fzr(
    str(TMPL),
    {"P": [0.5, 1.0, 1.5, 2.0, 2.5],
     "V": [10.0, 20.0, 30.0],
     "n": [1.0]},
    MODEL,
    results_dir=str(WORK / "results"),
    calculators=[CALC],
)
print(f"Shape: {df.shape}  (rows=cases, cols=inputs+outputs+metadata)")
print(df[["P", "V", "n", "temperature", "status"]].to_string(index=False))


  [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◣ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◤ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◥ [>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   0% (0/15) ETA: ...

◢ [██>░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]   6% (1/15) ETA: 7s

...

◣ [█████████████████████████████████████>░░]  93% (14/15) ETA: 0s

◤ [█████████████████████████████████████>░░]  93% (14/15) ETA: 0s

◥ [█████████████████████████████████████>░░]  93% (14/15) ETA: 0s

  [████████████████████████████████████████] 100% (15/15) Total: 7s

Shape: (15, 10)  (rows=cases, cols=inputs+outputs+metadata)
  P    V   n  temperature status
0.5 10.0 1.0    60.931026   done
0.5 20.0 1.0   121.862052   done
0.5 30.0 1.0   182.793078   done
1.0 10.0 1.0   121.862052   done
1.0 20.0 1.0   243.724104   done
1.0 30.0 1.0   365.586156   done
1.5 10.0 1.0   182.793078   done
1.5 20.0 1.0   365.586156   done
1.5 30.0 1.0   548.379235   done
2.0 10.0 1.0   243.724104   done
2.0 20.0 1.0   487.448209   done
2.0 30.0 1.0   731.172313   done
2.5 10.0 1.0   304.655130   done
2.5 20.0 1.0   609.310261   done
2.5 30.0 1.0   913.965391   done


## 11 · Visualise

In [12]:
df["P"] = df["P"].astype(float)
df["V"] = df["V"].astype(float)
df["temperature"] = pd.to_numeric(df["temperature"], errors="coerce")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for p in sorted(df["P"].unique()):
    sub = df[df["P"] == p].sort_values("V")
    ax1.plot(sub["V"], sub["temperature"], "o-", label=f"P={p} atm")
ax1.set_xlabel("Volume (L)"); ax1.set_ylabel("Temperature (K)")
ax1.set_title("T = PV/nR  — effect of volume"); ax1.legend(); ax1.grid(alpha=.3)

for v in sorted(df["V"].unique()):
    sub = df[df["V"] == v].sort_values("P")
    ax2.plot(sub["P"], sub["temperature"], "s-", label=f"V={v} L")
ax2.set_xlabel("Pressure (atm)"); ax2.set_ylabel("Temperature (K)")
ax2.set_title("T = PV/nR  — effect of pressure"); ax2.legend(); ax2.grid(alpha=.3)

plt.tight_layout()
plt.savefig(str(WORK / "perfectgas.png"), dpi=100)
plt.show()
print("Plot saved to", WORK / "perfectgas.png")


Plot saved to work_01_getting_started/perfectgas.png


/tmp/ipykernel_50478/3206548246.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12 · Sanity check — compare with analytical formula

In [13]:
R = 0.08206
df["T_theory"] = df["P"] * df["V"] / (df["n"].astype(float) * R)
df["error_K"] = abs(df["temperature"] - df["T_theory"])
print(df[["P", "V", "temperature", "T_theory", "error_K"]].to_string(index=False))
print(f"\nMax absolute error: {df['error_K'].max():.2e} K  ✓")


  P    V  temperature   T_theory      error_K
0.5 10.0    60.931026  60.931026 7.847916e-08
0.5 20.0   121.862052 121.862052 1.569583e-07
0.5 30.0   182.793078 182.793078 2.354375e-07
1.0 10.0   121.862052 121.862052 1.569583e-07
1.0 20.0   243.724104 243.724104 3.139166e-07
1.0 30.0   365.586156 365.586156 4.708750e-07
1.5 10.0   182.793078 182.793078 2.354375e-07
1.5 20.0   365.586156 365.586156 4.708750e-07
1.5 30.0   548.379235 548.379235 2.936875e-07
2.0 10.0   243.724104 243.724104 3.139166e-07
2.0 20.0   487.448209 487.448209 3.721667e-07
2.0 30.0   731.172313 731.172313 5.825007e-08
2.5 10.0   304.655130 304.655130 3.923959e-07
2.5 20.0   609.310261 609.310261 2.152083e-07
2.5 30.0   913.965391 913.965391 1.771875e-07

Max absolute error: 4.71e-07 K  ✓
